<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
### <center> المؤلف: إيليا لارشينكو، ODS Slack: ilya_l
    
## <center> البرنامج التعليمي
## <center> انسَ أمر GridSearch - كيفية ضبط المعلمات الفائقة باستخدام Hyperopt



## مقدمة



يعد ضبط المعلمات الفائقة جزءًا أساسيًا من أي مشروع للتعلم الآلي وواحدًا من أكثر المشاريع استهلاكًا للوقت.
حتى بالنسبة لأبسط النماذج، قد يستغرق الأمر ساعات للعثور على المعلمات المثالية، ناهيك عن الشبكات العصبية التي يمكن تحسينها يومًا أو أسابيع أو حتى لفترة أطول.
هناك طرق قياسية لحل هذه المهمة - بحث الشبكة والبحث العشوائي. كل عالم بيانات على دراية بها. ولكن هل هناك بدائل؟ هل هناك طرق للعثور على معلمات أفضل والقيام بذلك بشكل أسرع؟
الجواب هو نعم - ضبط المعلمات الفائقة ليس أكثر من مهمة تحسين الوظيفة. ومن الواضح أن الشبكة أو البحث العشوائي لا يبدو أنهما الخوارزميات الوحيدة والأفضل.
في هذا البرنامج التعليمي، سأفكر في طريقتين بديلتين - TPE والتليين المحاكي. هذه الأساليب ليست البدائل الوحيدة ولكنها عادةً ما تعمل بشكل أفضل من أساليب البحث القياسية كما أنها سهلة التنفيذ. سأصف كيفية عملها من الناحية النظرية وسأوضح لك كيفية استخدامها عمليًا باستخدام مكتبة Hyperopt.
بعد هذا البرنامج التعليمي، ستعرف كيفية تسريع عملية النمذجة الخاصة بك بسهولة.



## خطوة التحضير



لنقم باستيراد بعض المكتبات القياسية


In [ ]:
import numpy as np
import pandas as pd
from lightgbm.sklearn import LGBMRegressor
from sklearn.metrics import mean_squared_error

%matplotlib inline


سنعرض ونقارن بين الخوارزميات المختلفة لمجموعة بيانات مرض السكري من sklearn.datasets. دعونا تحميله.


In [ ]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()
n = diabetes.data.shape[0]

data = diabetes.data
targets = diabetes.target

يمكنك العثور على وصف مجموعة البيانات هنا: [https://www4.stat.ncsu.edu/~boos/var.select/diabetes.html]
    
القصة الطويلة باختصار: هذه هي مجموعة البيانات التي تحتوي على معلومات حول بعض المرضى والمقياس المستهدف "المقياس الكمي لتطور المرض بعد عام واحد من خط الأساس". ولأغراض هذا البرنامج التعليمي، لا تحتاج حتى إلى فهم البيانات، فقط ضع في اعتبارك أننا نقوم بحل بعض مشكلات الانحدار ونريد ضبط المعلمات الفائقة لدينا.



مجموعة البيانات صغيرة جدًا. لقد قمت باختياره لأنه سيكون من السهل توضيح المفهوم الأساسي باستخدامه. لن تحتاج إلى الانتظار لساعات عندما يتم حساب كل شيء. سنقوم بتقسيم مجموعة البيانات إلى أجزاء تدريب واختبار. سيتم تقسيم جزء القطار إلى شقين، وسوف نستخدم Cross Validation MSE كمقاييس نهائية نقوم وفقًا لها بتحسين المعلمات.
إخلاء المسؤولية: نموذج اللعبة هذا بعيد عن الواقع ويستخدم فقط للتوضيح السريع. نظرًا لصغر مجموعة البيانات وطياتها فقط، يمكن أن تكون غير مستقرة - تتغير النتائج بشكل كبير مع اختلاف الحالة العشوائية.


In [ ]:
from sklearn.model_selection import KFold, cross_val_score, train_test_split

random_state = 42
n_iter = 50

train_data, test_data, train_targets, test_targets = train_test_split(
    data, targets, test_size=0.20, shuffle=True, random_state=random_state
)

num_folds = 2
kf = KFold(n_splits=num_folds, random_state=random_state)


سنحاول حل المشكلة باستخدام LGBMRegressor. يحتوي Gradient Boosting على الكثير من المعلمات الفائقة التي يمكن تحسينها، ولهذا السبب يعد خيارًا جيدًا للعرض التوضيحي الذي نقدمه.


In [ ]:
model = LGBMRegressor(random_state=random_state)


دعونا ندرب نموذجًا أساسيًا باستخدام المعلمات الافتراضية:


In [ ]:
%%time
score = -cross_val_score(
    model, train_data, train_targets, cv=kf, scoring="neg_mean_squared_error", n_jobs=-1
).mean()
print(score)


نتيجة النموذج الجاهز هي 3532. دعونا نحاول تحسينه باستخدام أساليب التحسين المختلفة.
لأغراض العرض التوضيحي، سنقوم بتحسين ضبط النموذج من خلال 3 معلمات فقط:
- عدد المقدرين: من 100 إلى 2000
-العمق الأقصى: من 2 إلى 20
- معدل التعلم: من 10e-5 إلى 1



## بحث الشبكة


الطريقة الأولى والأبسط للتجربة هي GridSearchCV المضمنة في sklearn.model_selection
يقوم هذا الأسلوب فقط بتجربة مجموعات جميع المعلمات المتاحة 1 في 1 واختيار المجموعة التي تحتوي على أفضل نتائج التحقق من الصحة.
هذا النهج له عدة عيوب:
1. إنه بطيء جدًا - ما عليك سوى تجربة جميع مجموعات جميع المعلمات وسيستغرق الأمر الكثير من الوقت. أي معلمة إضافية للتغيير تضاعف عدد التكرارات التي تحتاج إلى إكمالها. تخيل أنك تضيف إلى شبكة المعلمات معلمة جديدة تحتوي على 10 قيم محتملة، يمكن أن يتبين أن هذه المعلمة لا معنى لها ولكن سيتم زيادة الوقت الحسابي 10 مرات.
2. يمكن أن يعمل فقط مع القيم المنفصلة. إذا كان الأمثل العالمي على n_estimators=550، ولكنك تقوم بإجراء GridSearchCV من 100 إلى 1000 مع الخطوة 100، فلن تصل أبدًا إلى النقطة المثالية.
3. عليك معرفة/تخمين التعريب التقريبي الأمثل لإتمام البحث في وقت معقول.
يمكنك التغلب على بعض هذه العيوب: يمكنك إجراء بحث في الشبكة عن المعلمة حسب المعلمة، أو استخدامها عدة مرات بدءًا من الشبكة العريضة بخطوات كبيرة وتضييق الحدود وتقليل أحجام الخطوات في أي تكرارات. لكنها ستظل مكثفة حسابيًا وطويلة جدًا.



دعونا نقدر الوقت اللازم لإجراء بحث الشبكة في حالتنا. لنفترض أننا نريد أن تتكون شبكتنا من 20 قيمة محتملة لـ "n_estimators" (100 إلى 2000)، و19 قيمة لـ "max_degree" (2 إلى 20)، و5 قيم لـ "learning_rate" (10e-4 إلى 0.1).
هذا يعني أننا بحاجة إلى حساب cross_val_score 20\*19\*5 = 1900 مرة. إذا استغرقت عملية حسابية واحدة حوالي 0.5-1.0 ثانية، فسيستمر بحث الشبكة لدينا لمدة تتراوح بين 15-30 دقيقة تقريبًا. إنه أكثر من اللازم بالنسبة لمجموعة البيانات التي تحتوي على 400 نقطة بيانات تقريبًا.لا نريد أن ننتظر طويلاً. نحن بحاجة إلى تضييق الفترات التي نريد تحليلها باستخدام هذه الطريقة. لقد تركت فقط 5\*8\*3=120 مجموعة. على جهاز الكمبيوتر الخاص بي يتم حسابه في 1,5 دقيقة.
دعونا نفعل الحسابات:


In [ ]:
%%time
from sklearn.model_selection import GridSearchCV

param_grid = {
    "learning_rate": np.logspace(-3, -1, 3),
    "max_depth": np.linspace(5, 12, 8, dtype=int),
    "n_estimators": np.linspace(800, 1200, 5, dtype=int),
    "random_state": [random_state],
}

gs = GridSearchCV(
    model,
    param_grid,
    scoring="neg_mean_squared_error",
    fit_params=None,
    n_jobs=-1,
    cv=kf,
    verbose=False,
)

gs.fit(train_data, train_targets)
gs_test_score = mean_squared_error(test_targets, gs.predict(test_data))


print("Best MSE {:.3f} params {}".format(-gs.best_score_, gs.best_params_))


لقد نجحنا في تحسين النتائج. لكنه قضى الكثير من الوقت في ذلك. دعونا ننظر كيف تغيرت معلماتنا من التكرار إلى التكرار:


In [ ]:
gs_results_df = pd.DataFrame(
    np.transpose(
        [
            -gs.cv_results_["mean_test_score"],
            gs.cv_results_["param_learning_rate"].data,
            gs.cv_results_["param_max_depth"].data,
            gs.cv_results_["param_n_estimators"].data,
        ]
    ),
    columns=["score", "learning_rate", "max_depth", "n_estimators"],
)
gs_results_df.plot(subplots=True, figsize=(10, 10))


يمكننا أن نرى على سبيل المثال أن الحد الأقصى للعمق هو المعلمة الأقل أهمية ولا يؤثر على النتيجة بشكل كبير. لكننا نبحث في أكثر من 8 قيم مختلفة للعمق الأقصى، ومع أي قيمة ثابتة نبحث عن المعلمات الأخرى. ومن الواضح مضيعة للوقت والموارد.
دعونا نجرب نهج البحث العشوائي الآن.



## بحث عشوائي



يعد البحث العشوائي أكثر فعالية في المتوسط من البحث الشبكي.
المزايا الرئيسية:
1. لا تقضي وقتًا في معايير لا معنى لها. في كل خطوة، يقوم البحث العشوائي بتنويع جميع المعلمات.
2. في المتوسط، يتم العثور على المعلمات الفرعية المثالية بشكل أسرع بكثير من البحث على الشبكة.
3. لا يقتصر الأمر على الشبكة عندما نقوم بتحسين المعلمات المستمرة.
العيوب:
1. قد لا يتم العثور على المعلمة العالمية المثالية على الشبكة.
2. جميع الخطوات مستقلة. وفي كل خطوة معينة، لا يستخدم أي معلومات حول النتائج التي تم جمعها حتى الآن. لكنها يمكن أن تكون مفيدة. على سبيل المثال، إذا وجدنا حلاً جيدًا، فقد يكون من المفيد البحث حوله للعثور على نقطة أفضل مقارنة بالنظر إلى متغيرات عشوائية تمامًا أخرى.



دعونا نحاول استخدام RandomizedSearchCV من sklearn.model_selection.
سنبدأ بمساحة واسعة جدًا من المعلمات ونقوم بـ 50 خطوة عشوائية فقط:


In [ ]:
from scipy.stats import randint
%%time
from sklearn.model_selection import RandomizedSearchCV

param_grid_rand = {
    "learning_rate": np.logspace(-5, 0, 100),
    "max_depth": randint(2, 20),
    "n_estimators": randint(100, 2000),
    "random_state": [random_state],
}

rs = RandomizedSearchCV(
    model,
    param_grid_rand,
    n_iter=n_iter,
    scoring="neg_mean_squared_error",
    fit_params=None,
    n_jobs=-1,
    cv=kf,
    verbose=False,
    random_state=random_state,
)

rs.fit(train_data, train_targets)

rs_test_score = mean_squared_error(test_targets, rs.predict(test_data))

print("Best MSE {:.3f} params {}".format(-rs.best_score_, rs.best_params_))


كما نرى، فإن النتائج أفضل بالفعل من GridSearchCV. لقد أمضينا وقتًا أقل وقمنا بإجراء بحث أكثر اكتمالاً. دعونا نلقي نظرة على التصور لدينا:

In [ ]:
rs_results_df = pd.DataFrame(
    np.transpose(
        [
            -rs.cv_results_["mean_test_score"],
            rs.cv_results_["param_learning_rate"].data,
            rs.cv_results_["param_max_depth"].data,
            rs.cv_results_["param_n_estimators"].data,
        ]
    ),
    columns=["score", "learning_rate", "max_depth", "n_estimators"],
)
rs_results_df.plot(subplots=True, figsize=(10, 10))


كما نرى، كل خطوة عشوائية تمامًا. إنه يساعد على عدم قضاء الوقت في المعلمات غير المفيدة، لكنه لا يستخدم المعلومات التي تم جمعها في الخطوات الأولى لتحسين نتائج الخطوات الأخيرة.



يمكننا تعديل البحث العشوائي بإضافة المزيد من الاهتمام إلى المجالات التي وجدنا فيها بالفعل حلولاً جيدة. هناك طرق مختلفة للقيام بذلك. سننظر في اثنين منهم: مقدر بارزن المهيكل على شكل شجرة والصلب المقلد.



## هايبروبت



سوف نستخدم مكتبة Hyperopt [https://github.com/hyperopt/hyperopt] للتعامل مع هذه الخوارزميات. إنها واحدة من المكتبات الأكثر شعبية لتحسين المعلمات الفائقة.



لتثبيت المكتبة، يمكنك استخدام النقطة أو conda (حسب البيئة الخاصة بك)


In [ ]:
!pip install hyperopt
#!conda install -c conda-forge hyperopt


أولاً، دعونا نستورد بعض الوظائف المفيدة من فرط الحركة:
- fmin - الوظيفة الرئيسية بالنسبة لنا، فهي ستقلل من وظائفنا
- tpe ويصلب - نهج التحسين
- حصان - تشمل توزيعات مختلفة للمتغيرات
- المحاكمات - تستخدم للتسجيل


In [ ]:
from hyperopt import Trials, anneal, fmin, hp, tpe


تختلف واجهة Hyperop.fmin عن الشبكة أو البحث العشوائي. أولا وقبل كل شيء نحن بحاجة إلى إنشاء وظيفة للتقليل.


In [ ]:
def gb_mse_cv(params, random_state=random_state, cv=kf, X=train_data, y=train_targets):
    # the function gest a set of variable parameters in "param"
    params = {
        "n_estimators": int(params["n_estimators"]),
        "max_depth": int(params["max_depth"]),
        "learning_rate": params["learning_rate"],
    }

    # we use this params to create a new LGBM Regressor
    model = LGBMRegressor(random_state=random_state, **params)

    # and then conduct the cross validation with the same folds as before
    score = -cross_val_score(
        model, X, y, cv=cv, scoring="neg_mean_squared_error", n_jobs=-1
    ).mean()

    return score


نحن مستعدون أخيرًا - لدينا دالة gb_mse_cv()، والتي سوف نقوم بتقليل المعلمات المختلفة: 'learning_rate'، 'max_deepth'، 'n_estimators'. لنبدأ بخوارزمية TPE.



### مقدر بارزين منظم على شكل شجرة



TPE هي خوارزمية افتراضية لـ Hyperopt. ويستخدم نهج بايزي للتحسين. في كل خطوة، تحاول بناء نموذج احتمالي للوظيفة واختيار المعلمات الواعدة للخطوة التالية. بشكل عام، تعمل هذه الأنواع من الخوارزميات على النحو التالي:1. قم بإنشاء نقطة أولية عشوائية ${x^*}$
2. احسب ${F(x^*)}$
3. باستخدام تاريخ التجارب حاول بناء نموذج الاحتمال الشرطي $P(F | x)$
4. اختر ${x_i}$ والذي وفقًا لـ $P(F | x)$ سيؤدي على الأرجح إلى ${F(x_i)}$ أفضل
5. احسب القيمة الحقيقية لـ ${F(x_i)}$
6. كرر الخطوات من 3 إلى 5 حتى يتم استيفاء أحد معايير التوقف، على سبيل المثال i > max_eval
يمكنك العثور على مزيد من المعلومات حول خوارزمية TPE معينة، على سبيل المثال، هنا [https://towardsdatascience.com/a-conceptual-explanation-of-bayesian-model-based-hyperparameter-optimization-for-machine-learning-b8172278050f] أو في مقالات أخرى. ولكن هذا خارج نطاق هذا البرنامج التعليمي.
دعونا نذهب إلى الممارسة.



استخدام fmin بسيط جدًا. نحتاج فقط إلى تحديد المساحة المحتملة لمعلماتنا واستدعاء الوظيفة.


In [ ]:
%%time

# possible values of parameters
space = {
    "n_estimators": hp.quniform("n_estimators", 100, 2000, 1),
    "max_depth": hp.quniform("max_depth", 2, 20, 1),
    "learning_rate": hp.loguniform("learning_rate", -5, 0),
}

# trials will contain logging information
trials = Trials()


best = fmin(
    fn=gb_mse_cv,  # function to optimize
    space=space,
    algo=tpe.suggest,  # optimization algorithm, hyperotp will select its parameters automatically
    max_evals=n_iter,  # maximum number of iterations
    trials=trials,  # logging
    rstate=np.random.RandomState(
        random_state
    ),  # fixing random state for the reproducibility
)

# computing the score on the test set
model = LGBMRegressor(
    random_state=random_state,
    n_estimators=int(best["n_estimators"]),
    max_depth=int(best["max_depth"]),
    learning_rate=best["learning_rate"],
)
model.fit(train_data, train_targets)
tpe_test_score = mean_squared_error(test_targets, model.predict(test_data))

print("Best MSE {:.3f} params {}".format(gb_mse_cv(best), best))


لقد تمكنا من إيجاد حل أفضل مقارنة بالبحث العشوائي.
دعونا نلقي نظرة على تصور العملية


In [ ]:
tpe_results = np.array(
    [
        [
            x["result"]["loss"],
            x["misc"]["vals"]["learning_rate"][0],
            x["misc"]["vals"]["max_depth"][0],
            x["misc"]["vals"]["n_estimators"][0],
        ]
        for x in trials.trials
    ]
)

tpe_results_df = pd.DataFrame(
    tpe_results, columns=["score", "learning_rate", "max_depth", "n_estimators"]
)
tpe_results_df.plot(subplots=True, figsize=(10, 10))


يمكننا أن نرى أن حركة المعلمات عشوائية تمامًا ولكن النتائج تصبح أفضل بمرور الوقت: لا توجد درجات سيئة للغاية بعد 25 تكرارًا ولكن عدد الحلول الجيدة يزداد. بدأت الخوارزمية في التنبؤ بحلول جيدة جدًا، باستخدام المعلومات من الخطوات السابقة.



### يصلب مقلد



التلدين المحاكاة يقلل من الوظيفة ${F(x)}$ (في حالة تحسين المعلمات الفائقة x - المعلمات، F() - وظيفة درجة التحقق من الصحة) على النحو التالي:
1. قم بإنشاء نقطة أولية عشوائية ${x^*}$
2. احسب ${F(x^*)}$
3. أنشئ ${x_i}$ بشكل عشوائي في بعض أحياء ${x*}$
4. احسب ${F(x_i)}$
5. قم بتحديث ${x*}$ وفقًا للقاعدة:
    إذا ${F(x_i)}<={F(x^*)}: {x^*} = {x_i}$ 
    آخر: ${x^*} = {x_i}$ مع احتمال $p=\exp\left(\dfrac{F({x^*})-F({x_i})}{T_i}\right)$
حيث ${T_i}$، تسمى درجة الحرارة بتسلسل يتناقص باستمراركرر الخطوات من 3 إلى 5 حتى يتم استيفاء أحد معايير الإيقاف:
- أنا> max_eval
- ${T_i} < {T_{min}}$



على الرغم من أن ${T_i}$ مرتفع، فإن الخوارزمية تنفذ الكثير من خطوات الاستكشاف (على غرار البحث العشوائي) حيث أن احتمال التحديث ${x^*}$ مرتفع حتى لو كان ${F(x_i)}>{F(x^*)}$
ولكن عندما أصبح T أقل، ركزت الخوارزمية على الاستغلال - فكل ${x_i}$ قريب من أحد أفضل الحلول التي تم العثور عليها حتى الآن.
في النهاية، باستخدام معلمات الخوارزميات الصحيحة، يمكن الوصول إلى توازن جيد بين الاستغلال/الاستكشاف ويمكن أن يؤدي إلى نتائج أفضل مقارنة بالبحث العشوائي. دعونا نتحقق من ذلك في مثال لعبتنا.



يمكنك محاولة تنفيذ إدراكك الخاص لخوارزمية التلدين المحاكية (وهي أسهل بكثير من TPE)، ولكن تم تنفيذها بالفعل في Hyperopt ويمكننا فقط تعيين معلمة "algo" لـ fmin على "anneal.suggest" (سيختار Hyperopt تلقائيًا معلمات التلدين لك).


In [ ]:
%%time

# possible values of parameters
space = {
    "n_estimators": hp.quniform("n_estimators", 100, 2000, 1),
    "max_depth": hp.quniform("max_depth", 2, 20, 1),
    "learning_rate": hp.loguniform("learning_rate", -5, 0),
}

# trials will contain logging information
trials = Trials()


best = fmin(
    fn=gb_mse_cv,  # function to optimize
    space=space,
    algo=anneal.suggest,  # optimization algorithm, hyperotp will select its parameters automatically
    max_evals=n_iter,  # maximum number of iterations
    trials=trials,  # logging
    rstate=np.random.RandomState(
        random_state
    ),  # fixing random state for the reproducibility
)

# computing the score on the test set
model = LGBMRegressor(
    random_state=random_state,
    n_estimators=int(best["n_estimators"]),
    max_depth=int(best["max_depth"]),
    learning_rate=best["learning_rate"],
)
model.fit(train_data, train_targets)
sa_test_score = mean_squared_error(test_targets, model.predict(test_data))

print("Best MSE {:.3f} params {}".format(gb_mse_cv(best), best))

In [ ]:
sa_results = np.array(
    [
        [
            x["result"]["loss"],
            x["misc"]["vals"]["learning_rate"][0],
            x["misc"]["vals"]["max_depth"][0],
            x["misc"]["vals"]["n_estimators"][0],
        ]
        for x in trials.trials
    ]
)

sa_results_df = pd.DataFrame(
    sa_results, columns=["score", "learning_rate", "max_depth", "n_estimators"]
)
sa_results_df.plot(subplots=True, figsize=(10, 10))


يوضح هذا التصور الفكرة الرئيسية لخوارزمية التلدين المحاكية بشكل جيد للغاية. في البداية، عندما تكون درجة الحرارة مرتفعة، يعمل بشكل مشابه للبحث العشوائي - فهو يقوم فقط باستكشاف جميع الحالات الممكنة. ولكن مع التبريد فإنه ينتقل إلى مرحلة الاستغلال والتركيز على المجالات الواعدة. وأخيرًا يتقارب مع الحل الجيد جدًا.



## النتائج



دعونا نرسم best_cumulative_score مقابل number_of_iterations لجميع الأساليب:


In [ ]:
scores_df = pd.DataFrame(index=range(n_iter))
scores_df["Grid Search"] = gs_results_df["score"].cummin()
scores_df["Random Search"] = rs_results_df["score"].cummin()
scores_df["TPE"] = tpe_results_df["score"].cummin()
scores_df["Annealing"] = sa_results_df["score"].cummin()

ax = scores_df.plot()

ax.set_xlabel("number_of_iterations")
ax.set_ylabel("best_cumulative_score")

يمكننا أن نرى أن خوارزميات TPE والتلدين تستمر فعليًا في تحسين نتائج البحث بمرور الوقت حتى في الخطوات اللاحقة، بينما وجد البحث العشوائي حلاً جيدًا تمامًا في البداية ثم أدى إلى تحسين النتائج بشكل طفيف فقط. الفرق الحالي بين نتائج TPE وRandomizedSearch صغير جدًا، ولكن في بعض تطبيقات الحياة الواقعية التي تحتوي على نطاق أكثر تنوعًا من المعلمات الفائقة، يمكن أن يمنحك تحسن كبير في الوقت/النتيجة.
ملاحظة: في الحياة الواقعية، من الأصح استخدام الوقت وليس عددًا من التكرارات للمقارنة، ولكن في مثال لعبتنا، تكون نسبة الوقت المستغرق في الحسابات الإضافية في tpe والتليين مرتفعة مقارنة بوقت حساب cross_val_score، لذلك قررت عدم تضليلك بشأن السرعة الحسابية لدرجات Hyperopt والمخطط فيما يتعلق برقم التكرار.



وللتأكد من أن كل شيء كان صحيحًا، فلنقارن نتائج بيانات الاختبار، ونتأكد من توافقها مع نتائج التحقق المتبادل


In [ ]:
print("Test MSE scored:")
print("Grid Search {:.3f}".format(gs_test_score))
print("Random Search {:.3f}".format(rs_test_score))
print("TPE {:.3f}".format(tpe_test_score))
print("Annealing {:.3f}".format(sa_test_score))


اتضح أن نتائج خوارزمية التلدين لديها في الواقع أدنى درجة اختبار. النتائج تهتز وفي المتوسط، تكون درجات الاختبار الأفضل مقارنة بالسيرة الذاتية هي نتائج مجموعة البيانات الصغيرة جدًا.



## السيرة الذاتية



أنت الآن تعرف المزيد عن أساليب تحسين المعلمات الفائقة المختلفة ويمكنك تطبيق مكتبة Hyperopt عمليًا. في الحالة الحقيقية، لن تعرف أبدًا مسبقًا أي نهج سيكون الأفضل (حتى في مثال اللعبة هذا مع بعض الحالات العشوائية التي يمكن أن يفوز فيها RandomizedSearch)، وفي بعض الأحيان يمكن أن يكون GridSearch السريع أو RandomizedSearch البسيط خيارًا جيدًا، ولكن من المفيد دائمًا معرفة البدائل.
نأمل أن يوفر لك هذا البرنامج التعليمي الكثير من الوقت في مشاريع ML المستقبلية.



##مكافأة


في الواقع، يحتوي hypeopt على أغلفة لوظائف sklearn الأكثر شيوعًا. باستخدامها، لا تحتاج حتى إلى تحديد مساحة المعلمات التي تحتاجها، فقط أخبر Hyperopt بالوظيفة التي تريد استخدامها. ما عليك سوى إلقاء نظرة على المثال أدناه (سنستخدم XGBoost كمثال لأنه لا يوجد LGBMRegressor في hpsklearn).


In [ ]:
# installing hpsklearn
!pip install hpsklearn

In [ ]:
%%time

from hpsklearn import HyperoptEstimator, xgboost_regression

estim = HyperoptEstimator(
    regressor=xgboost_regression("my_gb"),
    max_evals=n_iter,
    trial_timeout=60,
    seed=random_state,
)

estim.fit(train_data, train_targets)

print(mean_squared_error(test_targets, estim.predict(test_data)))


لم نفعل شيئًا تقريبًا، لكننا حصلنا على نتيجة جيدة. علاوة على ذلك، إذا كنت كسولًا بدرجة كافية، فلن تحتاج حتى إلى اختيار النموذج الذي سيحدده لك Hyperopt. فقط اطلب من Hyperopt العثور على أفضل مُقدِّر للبيانات المقدمة (لا تمرر معلمة "regressor" إلى HyperoptEstimator) وانتظر - يمكنك محاولة القيام بذلك بنفسك. مرحبًا بك في عالم AutoML :)